# Part 1 - Build the Detector Dataset

**P1 brick detector**

Builds the detector's training and evaluation data from the Gdańsk *Tagged images with LEGO bricks* dataset (Boiński, 2021):

1. Downloads the dataset (~6 GB, 2,933 real photos and 2,908 renders, each with a Pascal VOC `.xml` label file).
2. Reads every box from the real photos. All boxes become one class: `brick`.
3. Groups photos that were taken seconds apart (same scene) so near-duplicates never end up in both train and test.
4. Splits the groups 70/15/15 into train/dev/test, keeping photos with several bricks spread across all three.
5. Converts the labels to YOLO format, shrinks the photos to 1280 px on the long side, and saves `lego_det.zip` and the split list to Google Drive under `MyDrive/ece539-p1/`.

Only the real photos are used. The renders each show one brick on a plain background, which is not what the detector sees in practice.

## Settings

In [ ]:
from pathlib import Path

SEED = 539                  # same seed as P2 so the split is identical on every run
SPLIT = (0.70, 0.15, 0.15)  # train / dev / test share of the photos
GROUP_GAP_SECONDS = 60      # photos closer together than this count as the same scene
MAX_SIDE = 1280             # YOLO trains at 1280, so storing the photos any bigger is wasted space

DATASET_URL = ("https://mostwiedzy.pl/en/open-research-data/"
               "tagged-images-with-lego-bricks,202309140833448152311-0/download")
DATASET_BYTES = 6_016_987_324  # 6.0 GB
REPO_URL = "https://github.com/divijsinghkohli/cs539-backprop-boys.git"

WORK = Path("/content")
FULL_ZIP = WORK / "detector_data.zip"   # full download on Colab
DATA_DIR = WORK / "lego_det"            # converted YOLO dataset
REPO_DIR = WORK / "repo"
DRIVE_DIR = Path("/content/drive/MyDrive/ece539-p1")
DRIVE_RAW_ZIP = DRIVE_DIR / "raw" / "detector_data.zip"   # optional: put the zip here if the download is blocked
RESULTS_DIR = DRIVE_DIR / "results"

## Mount Drive and get detector.py from GitHub

In [ ]:
import importlib
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR / "P1") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "P1"))
import detector
importlib.reload(detector)

## Download the dataset

Downloads straight from the dataset page. If the site blocks the download, upload `detector_data.zip` to `MyDrive/ece539-p1/raw/` by hand and re-run this cell; it will copy it from there instead.

In [ ]:
import shutil

free_gb = shutil.disk_usage(WORK).free / 1e9
print(f"Free disk on this Colab machine: {free_gb:.0f} GB")

if FULL_ZIP.exists() and FULL_ZIP.stat().st_size == DATASET_BYTES:
    print("Already downloaded, skipping.")
elif DRIVE_RAW_ZIP.exists():
    print("Copying the zip from Drive...")
    shutil.copy(DRIVE_RAW_ZIP, FULL_ZIP)
else:
    assert free_gb > 10, "Not enough free disk for the 6 GB zip plus the converted dataset."
    get_ipython().system(
        f'wget -c -q --show-progress --progress=bar:force:noscroll -O "{FULL_ZIP}" "{DATASET_URL}"')

size = FULL_ZIP.stat().st_size
assert size == DATASET_BYTES, (
    f"Download incomplete: {size:,} of {DATASET_BYTES:,} bytes. Re-run this cell to resume.")
print(f"OK: {size / 1e9:.2f} GB")

## Read the labels

The zip has `photos/<n>/` and `renders/1/`, where `<n>` is the number of bricks in the photo.
Every `.jpg` has an `.xml` next to it with one `<object>` per brick.

In [ ]:
import zipfile
import pandas as pd
from pathlib import PurePosixPath

zf = zipfile.ZipFile(FULL_ZIP)
names = set(zf.namelist())
label_files = sorted(n for n in names if n.startswith("photos/") and n.endswith(".xml"))

rows = []
for xml_name in label_files:
    path = PurePosixPath(xml_name)
    width, height, boxes = detector.read_voc(zf.read(xml_name))
    image_name = str(path.with_suffix(".jpg"))
    assert image_name in names, f"No image for {xml_name}"
    assert len(boxes) == int(path.parent.name), f"{xml_name}: folder says {path.parent.name} bricks, label has {len(boxes)}"
    rows.append({"stem": path.stem, "zip_image": image_name, "n_bricks": len(boxes),
                 "width": width, "height": height, "boxes": boxes})

photos = pd.DataFrame(rows)
assert photos.stem.is_unique, "Two photos share a file name"
print(f"{len(photos):,} photos with {photos.n_bricks.sum():,} bricks")
photos.n_bricks.value_counts().sort_index().rename("photos").to_frame().T

## Group and split 70/15/15

The photos come from three sources, and each one names its files differently (phone `IMG_<date>_<time>`, sorter app `..._<millisecond timestamp>`, and `<part>_flash_<n>`).
Photos taken within `GROUP_GAP_SECONDS` of each other are put in the same group, and the flash photos are grouped by part number.
Whole groups go into one split, so the test set never contains a near-copy of a training photo.

Most photos only have one brick. Groups that contain any photo with 2+ bricks are split separately, so dev and test both get their share of busy scenes.

In [ ]:
photos["group"] = photos.stem.map(detector.group_photos(photos.stem.tolist(), GROUP_GAP_SECONDS))
photos["multi"] = photos.groupby("group").n_bricks.transform("max") > 1

photos["split"] = None
for multi, part in photos.groupby("multi"):
    sizes = part.groupby("group").size().to_dict()
    assignment = detector.split_groups(sizes, SPLIT, SEED)
    photos.loc[part.index, "split"] = part.group.map(assignment)

print(f"{photos.group.nunique()} groups, largest has {photos.groupby('group').size().max()} photos\n")
bricks_bucket = pd.cut(photos.n_bricks, [0, 1, 5, 100], labels=["1 brick", "2-5 bricks", "6+ bricks"])
summary = pd.crosstab(bricks_bucket, photos.split, margins=True)[["train", "dev", "test", "All"]]
print("Share of photos:", (summary.loc["All"] / len(photos)).round(3).to_dict())
summary

## Write the YOLO dataset

For each photo: shrink it so the long side is `MAX_SIDE`, save it to `images/<split>/`, and write `labels/<split>/<name>.txt` with one `0 cx cy w h` line per brick (fractions of the image size, so shrinking doesn't change them).

In [ ]:
import io
from PIL import Image
from tqdm.auto import tqdm

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
for split in ("train", "dev", "test"):
    (DATA_DIR / "images" / split).mkdir(parents=True)
    (DATA_DIR / "labels" / split).mkdir(parents=True)

for row in tqdm(photos.itertuples(), total=len(photos), desc="converting"):
    with Image.open(io.BytesIO(zf.read(row.zip_image))) as img:
        assert img.size == (row.width, row.height), f"{row.stem}: image size does not match its label"
        detector.resize_max_side(img.convert("RGB"), MAX_SIDE).save(
            DATA_DIR / "images" / row.split / f"{row.stem}.jpg", quality=95)
    lines = detector.to_yolo_lines(row.boxes, row.width, row.height)
    (DATA_DIR / "labels" / row.split / f"{row.stem}.txt").write_text("\n".join(lines) + "\n")

for split in ("train", "dev", "test"):
    n_img = len(list((DATA_DIR / "images" / split).glob("*.jpg")))
    n_lbl = len(list((DATA_DIR / "labels" / split).glob("*.txt")))
    print(f"{split:5s}: {n_img} images, {n_lbl} label files")

## Check a few labels by eye

Green boxes are drawn from the converted YOLO labels. If they don't sit on the bricks, something is wrong with the conversion.

In [ ]:
import matplotlib.pyplot as plt

sample = pd.concat([photos[photos.n_bricks == 1].sample(3, random_state=SEED),
                    photos[photos.n_bricks > 1].sample(3, random_state=SEED)])
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, row in zip(axes.flat, sample.itertuples()):
    img = Image.open(DATA_DIR / "images" / row.split / f"{row.stem}.jpg")
    gt = detector.read_yolo_labels(DATA_DIR / "labels" / row.split / f"{row.stem}.txt", *img.size)
    ax.imshow(detector.draw_boxes(img, [], gt))
    ax.set_title(f"{row.stem} ({row.n_bricks} bricks, {row.split})", fontsize=8)
    ax.axis("off")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "label_check.png", dpi=120)
plt.show()

## Save to Drive

Packs the converted dataset into one zip on Drive. Training copies this one file to the Colab VM at the start of each session.

In [ ]:
tmp = WORK / "lego_det.zip"
with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as z:   # JPEGs are already compressed
    for f in tqdm(sorted(DATA_DIR.rglob("*")), desc="zipping"):
        if f.is_file():
            z.write(f, f.relative_to(DATA_DIR))

out = DRIVE_DIR / "data" / "lego_det.zip"
out.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(tmp, out)
print(f"Saved {out}  ({out.stat().st_size / 1e9:.2f} GB)")

photos[["stem", "split", "group", "n_bricks", "width", "height"]].sort_values(["split", "stem"]).to_csv(
    RESULTS_DIR / "splits.csv", index=False)
summary.to_csv(RESULTS_DIR / "split_counts.csv")

## Export the result files

In [ ]:
from google.colab import files

archive = shutil.make_archive(str(WORK / "p1_data_results"), "zip", RESULTS_DIR)
print("Results:", sorted(p.name for p in RESULTS_DIR.iterdir() if p.is_file()))
files.download(archive)

## Outputs

* `MyDrive/ece539-p1/data/lego_det.zip`: the YOLO dataset (`images/` and `labels/` for train, dev, test), used by `02_train.ipynb`
* `MyDrive/ece539-p1/results/splits.csv`: which split every photo is in, its scene group and brick count
* `MyDrive/ece539-p1/results/split_counts.csv`: photos per split by number of bricks
* `MyDrive/ece539-p1/results/label_check.png`: converted labels drawn on sample photos

Expected: about 2,050 / 440 / 440 photos, with multi-brick photos in every split.